In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# Load Bronze table
# ---------------------------------------------------------

bronze = spark.table("financial_balance_sheets_bronze")

print("Rows:", bronze.count())
print("Columns:", len(bronze.columns))

display(bronze.limit(10))

In [0]:
# ---------------------------------------------------------
# Data Quality Scorecard
# ---------------------------------------------------------

required_columns = [
    "source",
    "dataset",
    "freq",
    "unit",
    "co_nco",
    "sector",
    "finpos",
    "na_item",
    "geo",
    "time",
    "value",
    "status"
]

# DQ-001: Required columns
missing_columns = [
    column for column in required_columns
    if column not in bronze.columns
]

rule_001 = (
    "PASS"
    if len(missing_columns) == 0
    else "FAIL"
)

# DQ-002: Dimension completeness
dimension_columns = [
    "freq",
    "unit",
    "co_nco",
    "sector",
    "finpos",
    "na_item",
    "geo",
    "time"
]

dimension_nulls = 0

for column in dimension_columns:
    dimension_nulls += bronze.filter(
        F.col(column).isNull()
    ).count()

rule_002 = (
    "PASS"
    if dimension_nulls == 0
    else "FAIL"
)

# DQ-003: Business key uniqueness
business_key = dimension_columns

duplicate_groups = (
    bronze
    .groupBy(*business_key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

rule_003 = (
    "PASS"
    if duplicate_groups == 0
    else "FAIL"
)

# DQ-004: Numeric validity
invalid_numeric = bronze.filter(
    F.col("value").isNotNull() &
    (
        F.isnan("value") |
        (F.col("value") == float("inf")) |
        (F.col("value") == float("-inf"))
    )
).count()

rule_004 = (
    "PASS"
    if invalid_numeric == 0
    else "FAIL"
)

# DQ-005: Source observation status
missing_flagged = bronze.filter(
    (F.col("value").isNull()) &
    (F.col("status") == "m")
).count()

provisional = bronze.filter(
    F.col("status") == "p"
).count()

rule_005 = "INFO"


# ---------------------------------------------------------
# Build scorecard
# ---------------------------------------------------------

scorecard = [
    (
        "DQ-001",
        "Schema",
        rule_001,
        f"Missing required columns: {len(missing_columns)}"
    ),
    (
        "DQ-002",
        "Dimension completeness",
        rule_002,
        f"Null dimension fields: {dimension_nulls}"
    ),
    (
        "DQ-003",
        "Uniqueness",
        rule_003,
        f"Duplicate business-key groups: {duplicate_groups}"
    ),
    (
        "DQ-004",
        "Numeric validity",
        rule_004,
        f"Invalid numeric observations: {invalid_numeric}"
    ),
    (
        "DQ-005",
        "Source status",
        rule_005,
        f"Source-flagged missing: {missing_flagged}; provisional: {provisional}"
    )
]

dq_scorecard = spark.createDataFrame(
    scorecard,
    [
        "rule_id",
        "dimension",
        "result",
        "details"
    ]
)

display(dq_scorecard)

In [0]:
# ---------------------------------------------------------
# Observation Status Register
# ---------------------------------------------------------

status_register = (
    bronze
    .filter(F.col("status").isNotNull())
    .select(
        "freq",
        "unit",
        "co_nco",
        "co_nco_label",
        "sector",
        "sector_label",
        "finpos",
        "finpos_label",
        "na_item",
        "na_item_label",
        "geo",
        "time",
        "value",
        "status"
    )
    .withColumn(
        "status_meaning",
        F.when(
            F.col("status") == "m",
            "Missing value; data cannot exist"
        )
        .when(
            F.col("status") == "p",
            "Provisional value"
        )
        .otherwise("Other source status")
    )
)

print("Flagged observations:", status_register.count())

display(status_register)